## 简答题

1. TensorFlow 是否可以简单替代 NumPy？两者之间的主要区别是什么？

2. 使用 `tf.range(10)` 和 `tf.constant(np.arange(10))` 是否会得到相同的结果？

3. 可以通过编写函数或继承 `tf.keras.losses.Loss` 来定义自定义损失函数。两种方法分别应该在什么时候使用？

4. 可以直接在函数中定义自定义指标或采用 `tf.keras.metrics.Metric` 子类。两种方法分别应该在什么时候使用？

5. 什么时候应该自定义层而不是自定义模型？

6. 有哪些示例需要编写自定义训练循环？

7. 自定义 Keras 组件中可以包含任意 Python 代码，还是必须转换为 TF 函数？

8. 如果要将函数转换为 TF 函数，应避免哪些主要模式？

9. 何时需要创建动态 Keras 模型？ 如何动态创建Keras模型？为什么不是所有模型都动态化？


 1.不可以。TensorFlow支持GPU，支持分布式计算，包含一种即时编译器可使其针对速度和内存使用情况来优化计算，计算图可以导出为可移植格式，它实现了反向模式的自动微分；Numpy主要用于数值计算支持cpu。

 2.会

 3.当自定义的损失函数比较简单时可以编写函数，当自定义损失函数比较复杂时，比如流失损失函数时，需要继承tf.keras.losses.Loss

 4.当自定义指标比较简单时可以直接在函数中定义，当自定义指标比较复杂，如需要可以累加指标时需要继承 tf.keras.metrics.Metric 子类

5.想构建一个架构，其中包含TensorFlow未提供默认实现的奇异层。或者可能只是想构建一个具有重复结构的架构（其中特定的层块重复多次），将每个块视为一个层会很方便。对于这种情况需要构建一个自定义层

6.宽深神经网络的论文使用了两种不同的优化器：一种用于宽路径，另一种用于深路径。由于fit()方法只使用一个优化器（在编译模型时指定的优化器），因此实现该论文需要编写自己的自定义循环

7.可以包含部分python代码，但是涉及到张量运算或者自动微分以及需要计算图时要转为TensorFlow函数

8.避免调用外部库，随机数生成必须使用 TF 提供的方法，避免副作用，慎用py_function，Python函数要符合规则，变量必须只创建一次，源码可用性，避免使用 Python 原生循环遍历 Dataset，出于性能原因，应尽可能使用向量化实现，而不是使用循环

9.当模型的结构需要根据输入数据的形状或其他运行时条件动态变化时，需要创建动态 Keras 模型。在函数中根据输入形状来决定添加多少个全连接层，或者使用循环来动态堆叠卷积层等。对于大多数常规的深度学习任务，模型结构在训练前是可以确定的，不需要动态调整，所以不需要都动态化。

## 编程题

1. 实现一个执行层归一化的自定义层：
    - a. `build()` 方法应定义两个可训练的权重 α 和 β，它们的形状均为 `input_shape[-1:]`，数据类型为 `tf.float32`。α 应该用 1 初始化，而 β 必须用 0 初始化。
    - b. `call()` 方法应计算每个实例特征的均值和标准差。为此，可以使用 `tf.nn.moments(inputs, axes=-1, keepdims=True)`，它返回同一实例的均值 μ 和方差 σ²（计算方差的平方根便可获得标准差）。然后，该函数应计算并返回
      $$
      \alpha \otimes \frac{(X-\mu)}{(\sigma+\epsilon)} + \beta
      $$
      其中 ε 是表示项精度的一个常量（避免被零除的小常数，例如 0.001）,$\otimes$表示逐个元素相乘
    - c. 确保自定义层产生与tf.keras.layers.LayerNormalization层相同（或几乎相同）的输出。

2. 使用自定义训练循环训练模型来处理Fashion MNIST数据集（13_神经网络介绍 里用的数据集）：

    - a.显示每个轮次、迭代、平均训练损失和每个轮次的平均精度（在每次迭代中更新），以及每个轮次结束时的验证损失和精度。
    - b.尝试对上面的层和下面的层使用具有不同学习率的不同优化器。

In [2]:
import tensorflow as tf
import numpy as np

class MyDense(tf.keras.layers.Dense):
    def __init__(self, units,activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)
    def build(self,input_shape):
        self.kernel_1 = self.add_weight(
            name="a", shape=[input_shape[-1], self.units],
            initializer="glorot_normal")
        self.kernel_2 = self.add_weight(
            name="b", shape=[input_shape[-1], self.units],
            initializer="glorot_normal")
        self.bias = self.add_weight(
            name="bias", shape=[self.units], initializer="zeros"
        )
        super().build(input_shape)
    def call(self,X):
        mean,variance2 = tf.nn.moments(X, axes=-1, keepdims=True)
        variance = tf.sqrt(variance2)
        return self.activation(((X-mean)/(variance + self.bias)) @ self.kernel_1 + self.kernel_2)
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "units": self.units, "activation": tf.keras.activations.serialize(self.activation)}

In [3]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist

# 保留训练集中的最后5000个图像进行验证
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

In [4]:
X_train = X_train.reshape(-1, 784)
X_valid = X_valid.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

In [5]:
class MyModel(tf.keras.Model):
    def __init__(self,**kwargs):
        super().__init__(**kwargs)
        self.lower_layer = tf.keras.layers.Dense(30, activation='relu')
        self.upper_layer = tf.keras.layers.Dense(10)
    def call(self,inputs):
        X = self.lower_layer(inputs)
        return self.upper_layer(X)
model = MyModel()

In [6]:
def random_batch(X,y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

# 定义一个函数，以显示训练状态，包括步数，步总数，从轮次开始以来的平均损失
def print_status_bar(step, total, loss, metrics=None):
    metrics = " - ".join([f"{m.name}: {m.result():.4f}" for m in [loss] + (metrics or [])])
    end = "" if step < total else "\n"

    print(f"\r{step}/{total} - " + metrics, end=end)

In [15]:
n_epochs = 5
batch_size = 32
n_steps = len(X_train) // batch_size

lower_optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
upper_optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# mean_loss = tf.keras.metrics.Mean(name="mean_loss")
# metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]

for epoch in range(1, n_epochs + 1):
    print(f"Epoch {epoch}/{n_epochs}")
    mean_loss = tf.keras.metrics.Mean(name="mean_loss")
    metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train, y_train)

        with tf.GradientTape(persistent=True) as tape:
            y_pred = model(X_batch, training=True)  # model.__call__() -> model.call()
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))  # tf.reduce_mean可以不需要

            # tf.add_n([main_loss, 其余的损失（正则化，layer自己加的）])  ->  main_loss + 其余的损失  -> 最终的损失
            loss = tf.add_n([main_loss] + model.losses)  # model.losses 可以把所有的损失 汇总一个python列表里

        gradients_lower = tape.gradient(loss, model.lower_layer.trainable_variables)
        gradients_upper = tape.gradient(loss, model.upper_layer.trainable_variables)
        del tape

        lower_optimizer.apply_gradients(zip(gradients_lower, model.lower_layer.trainable_variables))
        upper_optimizer.apply_gradients(zip(gradients_upper, model.upper_layer.trainable_variables))

        # 如果想给模型添加权重约束（kernel_constraint/bias_constraint) 在apply_gradients()之后立即应用这些约束
        for variable in model.variables:
            if variable.constraint is not None:
                variable.assign(variable.constraint(variable))

        mean_loss(loss)  # 这个批次的损失传给mean_loss, 返回这个轮次的平均损失
        for metric in metrics:
            metric(y_batch, y_pred)
        print_status_bar(step, n_steps, mean_loss, metrics)

Epoch 1/5
1718/1718 - mean_loss: 0.3244 - sparse_categorical_accuracy: 0.8813
Epoch 2/5
1718/1718 - mean_loss: 0.3160 - sparse_categorical_accuracy: 0.8837
Epoch 3/5
1718/1718 - mean_loss: 0.3072 - sparse_categorical_accuracy: 0.8869
Epoch 4/5
1718/1718 - mean_loss: 0.3051 - sparse_categorical_accuracy: 0.8880
Epoch 5/5
1718/1718 - mean_loss: 0.3041 - sparse_categorical_accuracy: 0.8872
